In [ ]:
Cella da eseguire solo la prima volta prima di creare la SparkSession: poi riavvare il kernel !!!

In [1]:
import os
import urllib.request
import subprocess

# Install Sedona Python package
subprocess.run(["pip", "install", "apache-sedona==1.7.0", "geopandas", "shapely", "--quiet"], 
               capture_output=True)
print("✅ Python packages installed")

jars = {
    "hadoop-aws-3.3.4.jar": "https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar",
    "aws-java-sdk-bundle-1.12.262.jar": "https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar",
     # Sedona core per Spark 3.5 + Scala 2.12
"sedona-spark-shaded-3.5_2.12-1.7.0.jar": "https://repo1.maven.org/maven2/org/apache/sedona/sedona-spark-shaded-3.5_2.12/1.7.0/sedona-spark-shaded-3.5_2.12-1.7.0.jar",    # Geotools per il supporto GeoJSON
"geotools-wrapper-1.6.1-28.2.jar": "https://repo1.maven.org/maven2/org/datasyslab/geotools-wrapper/1.6.1-28.2/geotools-wrapper-1.6.1-28.2.jar"}
for filename, url in jars.items():
    dest = f"/tmp/spark_jars/{filename}"
    if not os.path.exists(dest):
        print(f"⬇️  Downloading {filename}...")
        urllib.request.urlretrieve(url, dest)
        print(f"✅ {filename}")
    else:
        print(f"⏭️  Already exists: {filename}")

✅ Python packages installed
⏭️  Already exists: hadoop-aws-3.3.4.jar
⏭️  Already exists: aws-java-sdk-bundle-1.12.262.jar
⏭️  Already exists: sedona-spark-shaded-3.5_2.12-1.7.0.jar
⏭️  Already exists: geotools-wrapper-1.6.1-28.2.jar


In [2]:
# Imposta parametri di avvio di PySpark PRIMA della creazione della SparkSession
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--jars /tmp/spark_jars/hadoop-aws-3.3.4.jar,"
    "/tmp/spark_jars/aws-java-sdk-bundle-1.12.262.jar,"
    "/tmp/spark_jars/sedona-spark-shaded-3.5_2.12-1.7.0.jar,"
    "/tmp/spark_jars/geotools-wrapper-1.6.1-28.2.jar "
    "pyspark-shell"
)
print("✅ Environment set")

✅ Environment set


In [3]:


from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, substring, regexp_replace, trim, avg,
    when, round as spark_round, count,
    min as spark_min, max as spark_max, sum as spark_sum
)

BRONZE_PATH = "s3a://rental-observatory/bronze/"
SILVER_PATH = "s3a://rental-observatory/silver/"
GOLD_PATH   = "s3a://rental-observatory/gold/"

spark = SparkSession.builder \
    .appName("NYC_Rental_Stress") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f" Spark {spark.version}\n")


 Spark 3.5.0



In [4]:


# NYC ZIP code prefixes (5 boroughs only)
# Excludes Long Island, Hamptons, Upstate NY
NYC_ZIP_PREFIXES = ["100", "101", "102", "103", "104",  # Manhattan + Bronx
                    "111", "112", "113", "114", "116",  # Brooklyn + Queens
                    "103"]                               # Staten Island

# ============================================================
# 2. LOAD RAW DATA FROM BRONZE LAYER
# ============================================================
print("=" * 55)
print(" STEP 1 — LOADING DATA (Bronze → Spark)")
print("=" * 55)

pop_df = spark.read.csv(
    BRONZE_PATH + "ACSDT5Y2024.B01003-Data.csv",
    header=True, inferSchema=True)

income_df = spark.read.csv(
    BRONZE_PATH + "ACSDT5Y2024.B19013-Data.csv",
    header=True, inferSchema=True)

zillow_df = spark.read.csv(
    BRONZE_PATH + "Zip_zori_uc_sfrcondomfr_sm_month.csv",
    header=True, inferSchema=True)

listings_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .load(BRONZE_PATH + "listings_NY.csv.gz")

calendar_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .load(BRONZE_PATH + "calendar_NY.csv.gz")

print(f"  Population:  {pop_df.count():,} rows")
print(f"  Income:      {income_df.count():,} rows")
print(f"  Zillow:      {zillow_df.count():,} rows")
print(f"  Listings:    {listings_df.count():,} rows")
print(f"  Calendar:    {calendar_df.count():,} rows")
print("✅ Bronze loaded.\n")

 STEP 1 — LOADING DATA (Bronze → Spark)
  Population:  33,773 rows
  Income:      33,773 rows
  Zillow:      7,716 rows
  Listings:    36,261 rows
  Calendar:    13,235,283 rows
✅ Bronze loaded.



In [5]:
pop_df.printSchema()
pop_df.show(5)
pop_df.dtypes

# Popolazione per ZIP
census_population = (
    pop_df  # parto dal DataFrame originale (bronze)
    .filter(col("GEO_ID") != "Geography")  # rimuovo la riga header duplicata/non valida
    .withColumn(
        "zip_code",
        substring(col("GEO_ID"), -5, 5)  # estraggo gli ultimi 5 caratteri → ZIP code
    )
    .withColumn(
        "total_population",  # nuova colonna con la popolazione totale
        when(
            regexp_replace(col("B01003_001E"), "[^0-9]", "") == "",  # se dopo pulizia non resta nulla
            None  # allora metto valore nullo
        ).otherwise(
            regexp_replace(col("B01003_001E"), "[^0-9]", "")  # rimuovo tutto ciò che non è numero
            .cast("int")  # converto a intero
        )
    )
    .select("zip_code", "total_population")  # tengo solo le colonne utili
    .dropna()  # elimino righe con valori nulli
)
census_population.show()

root
 |-- GEO_ID: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- B01003_001E: string (nullable = true)
 |-- B01003_001M: string (nullable = true)
 |-- _c4: string (nullable = true)

+--------------+--------------------+---------------+--------------------+----+
|        GEO_ID|                NAME|    B01003_001E|         B01003_001M| _c4|
+--------------+--------------------+---------------+--------------------+----+
|     Geography|Geographic Area Name|Estimate!!Total|Margin of Error!!...|NULL|
|860Z200US00601|         ZCTA5 00601|          16669|                 510|NULL|
|860Z200US00602|         ZCTA5 00602|          37233|                 270|NULL|
|860Z200US00603|         ZCTA5 00603|          48448|                1035|NULL|
|860Z200US00606|         ZCTA5 00606|           5163|                 293|NULL|
+--------------+--------------------+---------------+--------------------+----+
only showing top 5 rows

+--------+----------------+
|zip_code|total_population

In [6]:
income_df.printSchema()

# Reddito mediano per ZIP
census_income = (
    income_df  # parto dal DataFrame originale (bronze)
    .filter(col("GEO_ID") != "Geography")  # rimuovo la riga header duplicata/non valida
    .withColumn("zip_code", substring(col("GEO_ID"), -5, 5))  # estraggo ZIP code
    .withColumn("median_income",  # nuova colonna reddito mediano
        when(regexp_replace(col("B19013_001E"), "[^0-9]", "") == "", None)  # se vuoto → null
        .otherwise(regexp_replace(col("B19013_001E"), "[^0-9]", "").cast("int"))  # pulisco e casto a int
    )
    .select("zip_code", "median_income")  # tengo solo colonne utili
    .dropna()  # rimuovo righe con null
)
census_income.show(5)

root
 |-- GEO_ID: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- B19013_001E: string (nullable = true)
 |-- B19013_001M: string (nullable = true)
 |-- _c4: string (nullable = true)

+--------+-------------+
|zip_code|median_income|
+--------+-------------+
|   00601|        19454|
|   00602|        21420|
|   00603|        20933|
|   00606|        20992|
|   00610|        24496|
+--------+-------------+
only showing top 5 rows



In [8]:
zillow_df.printSchema()
# Zillow ZORI — affitto di mercato NY
zillow_rent = (
    zillow_df
    .filter(col("State") == "NY")  # tengo solo lo stato New York
    .select(
        col("RegionName").cast("string").alias("zip_code"),  # ZIP code (cast a string + rename)
        col("2025-12-31").cast("float").alias("market_rent")  # affitto stimato a fine 2025
    )
    .dropna()  # rimuovo righe con valori null
)
zillow_rent.show(5)

root
 |-- RegionID: integer (nullable = true)
 |-- SizeRank: integer (nullable = true)
 |-- RegionName: integer (nullable = true)
 |-- RegionType: string (nullable = true)
 |-- StateName: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Metro: string (nullable = true)
 |-- CountyName: string (nullable = true)
 |-- 2015-01-31: double (nullable = true)
 |-- 2015-02-28: double (nullable = true)
 |-- 2015-03-31: double (nullable = true)
 |-- 2015-04-30: double (nullable = true)
 |-- 2015-05-31: double (nullable = true)
 |-- 2015-06-30: double (nullable = true)
 |-- 2015-07-31: double (nullable = true)
 |-- 2015-08-31: double (nullable = true)
 |-- 2015-09-30: double (nullable = true)
 |-- 2015-10-31: double (nullable = true)
 |-- 2015-11-30: double (nullable = true)
 |-- 2015-12-31: double (nullable = true)
 |-- 2016-01-31: double (nullable = true)
 |-- 2016-02-29: double (nullable = true)
 |-- 2016-03-31: double (nullable = true)
 |-- 

In [9]:
print(f"  Population silver: {census_population.count():,} ZIP")
print(f"  Income silver:     {census_income.count():,} ZIP")
print(f"  Zillow silver:     {zillow_rent.count():,} ZIP")

  Population silver: 33,772 ZIP
  Income silver:     30,547 ZIP
  Zillow silver:     296 ZIP


In [8]:
# ============================================================
# 4. SILVER — AIRBNB LISTINGS
#
# price e estimated_revenue_l365d sono NULL in tutto il dataset NY.
# Usiamo estimated_occupancy_l365d per calcolare l'occupancy_rate.
# Il prezzo sarà approssimato con Zillow ZORI nel Gold layer.
# ============================================================
print("\n--- Airbnb Silver ---")

airbnb_silver = (
    listings_df
    # Rimuove righe dove l'ID non è un numero valido (righe corrotte)
    .filter(col("id").cast("bigint").isNotNull())
    # Rimuove listing senza borough (non possiamo geolocalizzarli)
    .filter(col("neighbourhood_group_cleansed").isNotNull())
    # Rimuove listing senza coordinate (non mappabili)
    .filter(col("latitude").isNotNull())
    .filter(col("longitude").isNotNull())

    # Crea colonna float dei giorni occupati negli ultimi 365 giorni
    # Es: "240" (stringa) → 240.0 (float)
    .withColumn(
        "occupancy_days",
        col("estimated_occupancy_l365d").cast("float")
    )

    # 🛠 FIX 1: Calcolo diretto senza when
    # Spark gestisce già i NULL automaticamente (NULL → NULL)
    # Es: 240 giorni occupati → 240/365 = 0.6575
    .withColumn(
        "occupancy_rate",
        spark_round(col("occupancy_days") / 365.0, 4)
    )

    # 🛠 FIX 2: Convertiamo gli 0.0 sospetti in NULL
    # Motivo: nel dataset, 0 spesso significa "dato mancante"
    # Evita di distorcere medie e analisi nel Gold layer
    .withColumn(
        "occupancy_rate",
        when(col("occupancy_rate") == 0, None)
        .otherwise(col("occupancy_rate"))
    )

    .select(
        col("id").cast("bigint"),                          # ID univoco del listing
        "neighbourhood_group_cleansed",                    # Borough: Manhattan, Brooklyn, Queens, Bronx, Staten Island
        "neighbourhood_cleansed",                          # Quartiere specifico: es. "Williamsburg"
        col("latitude").cast("float"),                     # Latitudine per la heatmap
        col("longitude").cast("float"),                    # Longitudine per la heatmap
        "occupancy_rate",                                  # Tasso occupazione 0.0→1.0 (NULL = dato mancante)
        "occupancy_days",                                  # Giorni occupati raw
        col("room_type"),                                  # Entire home / Private room / etc.
        col("accommodates").cast("int"),                   # Numero massimo ospiti
        col("calculated_host_listings_count").cast("int")  # Listings per host → proxy host commerciale
    )
)

airbnb_count = airbnb_silver.count()
print(f"  Listings Airbnb (silver): {airbnb_count:,}")
print("✅ Airbnb Silver completato.\n")

airbnb_silver.select("neighbourhood_group_cleansed","occupancy_rate").show()


--- Airbnb Silver ---
  Listings Airbnb (silver): 36,445
✅ Airbnb Silver completato.

+----------------------------+--------------+
|neighbourhood_group_cleansed|occupancy_rate|
+----------------------------+--------------+
|                   Manhattan|          NULL|
|                    Brooklyn|        0.6575|
|                   Manhattan|        0.1644|
|                   Manhattan|        0.6575|
|                    Brooklyn|        0.6986|
|                    Brooklyn|          NULL|
|                    Brooklyn|          NULL|
|                   Manhattan|          NULL|
|                    Brooklyn|        0.1644|
|                   Manhattan|          NULL|
|                   Manhattan|          NULL|
|                   Manhattan|          NULL|
|                   Manhattan|          NULL|
|                    Brooklyn|        0.1644|
|                       Bronx|          NULL|
|                    Brooklyn|          NULL|
|                   Manhattan|         

In [10]:
# Verifica distribuzione prezzi nel calendar
calendar_df.select("price").show(10)

# Conta valori null e zero
from pyspark.sql.functions import count, when, col

calendar_df.select(
    count("price").alias("total_rows"),
    count(when(col("price").isNull(), 1)).alias("null_prices"),
    count(when(col("price") == "", 1)).alias("empty_prices"),
    count(when(col("price") == "$0.00", 1)).alias("zero_prices"),
).show()

+-----+
|price|
+-----+
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
+-----+
only showing top 10 rows

+----------+-----------+------------+-----------+
|total_rows|null_prices|empty_prices|zero_prices|
+----------+-----------+------------+-----------+
|         0|   13235283|           0|          0|
+----------+-----------+------------+-----------+



In [11]:
listings_df.select("price").show(10)


+-----+
|price|
+-----+
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
+-----+
only showing top 10 rows



In [ ]:
# ============================================================
# 4b. SILVER — CALENDAR
# ============================================================
print("\n--- Calendar Silver ---")

from pyspark.sql.functions import to_date

calendar_silver = (
    calendar_df
    # Rimuove righe con listing_id non valido
    .filter(col("listing_id").cast("bigint").isNotNull())
    # Cast a bigint per poter fare JOIN con airbnb_silver.id
    .withColumn("listing_id", col("listing_id").cast("bigint"))
    # Converte la stringa data "2025-12-06" in tipo Date di Spark
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
    # 1 se il listing è disponibile quel giorno, 0 altrimenti
    .withColumn(
        "is_available",
        when(col("available") == "t", 1).otherwise(0)
    )
    # NON disponibile (occupato + blocchi host)
    .withColumn(
        "is_blocked",
        when(col("available") == "f", 1).otherwise(0)
    )
    .select("listing_id", "date", "is_available", "is_blocked")
)

# Aggrega per listing
calendar_agg = (
    calendar_silver
    .groupBy("listing_id")
    .agg(
        spark_sum("is_available").alias("days_available"),
        spark_sum("is_blocked").alias("days_blocked"),
        count("date").alias("total_days")
    )
    .filter(col("total_days") >= 300)
    .withColumn(
        "cal_occupancy_rate",
        spark_round(col("days_blocked") / col("total_days"), 4)
    )
)

# Statistiche finali
airbnb_count = airbnb_silver.count()
cal_count = calendar_agg.count()

print(f"  Listings Airbnb (silver):    {airbnb_count:,}")
print(f"  Listing con dati calendar:   {cal_count:,}")

airbnb_silver.select(
    spark_round(avg("occupancy_rate"), 3).alias("avg_occupancy_rate"),
    spark_min("occupancy_rate").alias("min"),
    spark_max("occupancy_rate").alias("max")
).show()

print("✅ Calendar Silver completato.\n")

In [ ]:
calendar_agg.show(5)

In [ ]:
calendar_silver.show(5)

In [ ]:
calendar_df.show(5)

In [ ]:
"""# ============================================================
# 4c. SILVER — AIRBNB LISTINGS ENRICHED
#
# JOIN tra airbnb_silver e calendar_agg per arricchire
# ogni listing con i dati reali di occupazione dal calendar.
# Usiamo left join per mantenere tutti i listing anche se
# non hanno dati nel calendar (cal_occupancy_rate → NULL)
# ============================================================
print("\n--- Airbnb Listings Enriched ---")

airbnb_listings_enriched = (
    airbnb_silver
    # Left join: teniamo tutti i listing di airbnb_silver
    # anche se non hanno dati nel calendar
    .join(calendar_agg, airbnb_silver["id"] == calendar_agg["listing_id"], "left")
    # Usiamo cal_occupancy_rate dal calendar se disponibile,
    # altrimenti fallback su occupancy_rate da estimated_occupancy_l365d
    .withColumn(
        "final_occupancy_rate",
        when(col("cal_occupancy_rate").isNotNull(), col("cal_occupancy_rate"))
        .otherwise(col("occupancy_rate"))
    )
    .select(
        col("id"),
        "neighbourhood_group_cleansed",              # Borough
        "neighbourhood_cleansed",                    # Quartiere
        col("latitude"),
        col("longitude"),
        "room_type",
        "accommodates",
        "calculated_host_listings_count",            # Proxy host commerciale
        "occupancy_rate",                            # Stima Inside Airbnb
        "cal_occupancy_rate",                        # Dato reale dal calendar
        "final_occupancy_rate",                      # Quello che useremo nel Gold
        "days_available",                            # Giorni liberi nell'anno
        "days_blocked",                              # Giorni occupati/bloccati
        "total_days"                                 # Totale giorni nel dataset
    )
)

enriched_count = airbnb_listings_enriched.count()
print(f"  Listings enriched: {enriched_count:,}")
airbnb_listings_enriched.select(
    "neighbourhood_group_cleansed",
    "final_occupancy_rate",
    "cal_occupancy_rate",
    "occupancy_rate"
).show(5)
print("✅ Airbnb Listings Enriched completato.\n")"""

In [ ]:
# Prendi 10 ID da airbnb_silver
airbnb_ids = [row.id for row in airbnb_silver.select("id").limit(10).collect()]
print("Airbnb IDs:", airbnb_ids)

# Cerca quegli stessi ID nel calendar_agg
calendar_agg.filter(col("listing_id").isin(airbnb_ids)).show()

In [ ]:
# Quando è stato scrapeato il calendar?
calendar_df.select("date").distinct().orderBy("date").show(5)

# Quando è stato scrapeato il listings?
listings_df.select("last_scraped").distinct().show(5)

In [ ]:
# Quanti ID ha il calendar?
cal_ids = calendar_df.select("listing_id").distinct().count()
listing_ids = listings_df.select("id").distinct().count()
print(f"Calendar IDs: {cal_ids:,}")
print(f"Listings IDs: {listing_ids:,}")

# Quanti in comune?
cal_df2 = calendar_df.select(col("listing_id").cast("bigint").alias("id"))
common = cal_df2.join(listings_df.select("id"), "id", "inner")
print(f"ID in comune: {common.distinct().count():,}")

In [9]:
# ============================================================
# AIRBNB LISTINGS ENRICHED
#
# Il calendar disponibile non è allineato con i listings
# (0 ID in comune tra i due dataset).
# Si utilizza estimated_occupancy_l365d come proxy
# dell'occupancy rate, fornito direttamente da Inside Airbnb.
# ============================================================

airbnb_listings_enriched = (
    airbnb_silver
    .withColumnRenamed("occupancy_rate", "final_occupancy_rate")
)

print(f"  Listings enriched: {airbnb_listings_enriched.count():,}")
print("✅ Airbnb Listings Enriched completato.\n")

  Listings enriched: 36,445
✅ Airbnb Listings Enriched completato.



In [12]:
# ============================================================
# 4. SILVER — AIRBNB (CALENDAR + LISTINGS JOIN)
#
# Ora che calendar e listings sono allineati (dicembre 2025)
# possiamo fare il JOIN e calcolare l'occupancy rate reale
# dal calendar invece di usare la stima estimated_occupancy_l365d.
#
# Il calendar con 13M righe è il dataset Big Data principale
# che giustifica l'uso di Apache Spark.
# ============================================================
print("\n--- Airbnb Listings Enriched ---")

from pyspark.sql.functions import to_date

# ---- 4.1 Pulizia Listings ----
# Teniamo solo le colonne geografiche e demografiche
# che servono per il JOIN e lo spatial join successivo
listings_clean = (
    listings_df
    .filter(col("id").cast("bigint").isNotNull())
    .filter(col("neighbourhood_group_cleansed").isNotNull())
    .filter(col("latitude").isNotNull())
    .filter(col("longitude").isNotNull())
    .select(
        col("id").cast("bigint").alias("listing_id"),
        "neighbourhood_group_cleansed",
        "neighbourhood_cleansed",
        col("latitude").cast("float"),
        col("longitude").cast("float"),
        col("room_type"),
        col("accommodates").cast("int"),
        col("calculated_host_listings_count").cast("int")
    )
)

print(f"  Listings puliti: {listings_clean.count():,}")

# ---- 4.2 Pulizia e aggregazione Calendar ----
# Il calendar ha 13M righe — una per ogni giorno per ogni listing
# available = 't' → disponibile, 'f' → occupato/bloccato
calendar_agg = (
    calendar_df
    .filter(col("listing_id").cast("bigint").isNotNull())
    .withColumn("listing_id", col("listing_id").cast("bigint"))
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
    # 1 se disponibile, 0 altrimenti
    .withColumn(
        "is_available",
        when(col("available") == "t", 1).otherwise(0)
    )
    # 1 se occupato/bloccato, 0 altrimenti
    .withColumn(
        "is_blocked",
        when(col("available") == "f", 1).otherwise(0)
    )
    # Aggrega per listing: riduce da 13M a ~36k righe
    .groupBy("listing_id")
    .agg(
        spark_sum("is_available").alias("days_available"),
        spark_sum("is_blocked").alias("days_blocked"),
        count("date").alias("total_days")
    )
    # Tieni solo listing con almeno 300 giorni di dati
    .filter(col("total_days") >= 300)
    .withColumn(
        # Occupancy rate reale: giorni occupati / totale giorni
        "cal_occupancy_rate",
        spark_round(col("days_blocked") / col("total_days"), 4)
    )
    # Converti 0.0 in NULL — listing mai occupati non hanno dato affidabile
    .withColumn(
        "cal_occupancy_rate",
        when(col("cal_occupancy_rate") == 0, None)
        .otherwise(col("cal_occupancy_rate"))
    )
)

print(f"  Listing con dati calendar: {calendar_agg.count():,}")

# ---- 4.3 JOIN Calendar + Listings ----
# Inner join: teniamo solo listing presenti in entrambi i dataset
airbnb_listings_enriched = (
    calendar_agg
    .join(listings_clean, "listing_id", "inner")
    .select(
        "listing_id",
        "neighbourhood_group_cleansed",
        "neighbourhood_cleansed",
        "latitude",
        "longitude",
        "room_type",
        "accommodates",
        "calculated_host_listings_count",
        # Occupancy rate reale dal calendar
        # Sostituisce estimated_occupancy_l365d che era una stima
        col("cal_occupancy_rate").alias("final_occupancy_rate"),
        "days_available",
        "days_blocked",
        "total_days"
    )
)

# Statistiche finali
total = airbnb_listings_enriched.count()
with_occupancy = airbnb_listings_enriched \
    .filter(col("final_occupancy_rate").isNotNull()).count()

print(f"  Listings enriched:        {total:,}")
print(f"  Con occupancy valida:     {with_occupancy:,}")

airbnb_listings_enriched.select(
    spark_round(avg("final_occupancy_rate"), 3).alias("avg_occupancy_rate"),
    spark_min("final_occupancy_rate").alias("min"),
    spark_max("final_occupancy_rate").alias("max")
).show()

airbnb_listings_enriched.select(
    "neighbourhood_group_cleansed",
    "final_occupancy_rate"
).show(5)

print("✅ Airbnb Listings Enriched completato.\n")


--- Airbnb Listings Enriched ---
  Listings puliti: 36,261
  Listing con dati calendar: 36,261
  Listings enriched:        36,261
  Con occupancy valida:     32,996
+------------------+------+---+
|avg_occupancy_rate|   min|max|
+------------------+------+---+
|             0.595|0.0027|1.0|
+------------------+------+---+

+----------------------------+--------------------+
|neighbourhood_group_cleansed|final_occupancy_rate|
+----------------------------+--------------------+
|                   Manhattan|              0.7726|
|                    Brooklyn|                NULL|
|                   Manhattan|              0.6548|
|                    Brooklyn|              0.1507|
|                    Brooklyn|              0.0082|
+----------------------------+--------------------+
only showing top 5 rows

✅ Airbnb Listings Enriched completato.



In [13]:
# ============================================================
# 5. SAVE SILVER LAYER TO MINIO
#
# Salviamo i dataframe puliti in formato Parquet su MinIO.
# Parquet è un formato colonnare ottimizzato per Spark:
# - compressione migliore dei CSV
# - lettura più veloce (legge solo le colonne necessarie)
# - mantiene i tipi di dato (no problemi di cast al prossimo load)
# ============================================================
print("=" * 55)
print("💾 SAVING SILVER LAYER TO MINIO")
print("=" * 55)

# Census Population Silver
census_population.write \
    .mode("overwrite") \
    .parquet(SILVER_PATH + "census_population/")
print("✅ census_population saved")

# Census Income Silver
census_income.write \
    .mode("overwrite") \
    .parquet(SILVER_PATH + "census_income/")
print("✅ census_income saved")

# Zillow Rent Silver
zillow_rent.write \
    .mode("overwrite") \
    .parquet(SILVER_PATH + "zillow_rent/")
print("✅ zillow_rent saved")

airbnb_listings_enriched.write \
    .mode("overwrite") \
    .parquet(SILVER_PATH + "airbnb_listings_enriched/")
print("✅ airbnb_listings_enriched saved")

print("\n🎉 Silver layer saved to MinIO!\n")

💾 SAVING SILVER LAYER TO MINIO
✅ census_population saved
✅ census_income saved
✅ zillow_rent saved
✅ airbnb_listings_enriched saved

🎉 Silver layer saved to MinIO!



In [ ]:
# ============================================================
# 5. GOLD — MARKET ANALYSIS
#
# Il Gold layer combina tutti i dataset Silver per produrre
# le metriche finali che alimentano la dashboard.
# Tre tabelle Gold:
#   1. market_rental_stress: stress economico per ZIP code
#   2. airbnb_borough_summary: concentrazione Airbnb per borough
#   3. airbnb_vs_market: confronto Airbnb vs affitto di mercato
# ============================================================
print("=" * 55)
print("🏆 STEP 3 — GOLD")
print("=" * 55)


In [14]:
# ============================================================
# 5.1 GOLD — ECONOMIC PROFILE PER ZIP CODE
#
# Uniamo i tre dataset Silver (Census Population, Census Income, Zillow)
# usando zip_code come chiave di JOIN.
# Il risultato è un profilo economico completo per ogni ZIP code di NY.
# ============================================================
print("=" * 55)
print("🏆 STEP 3 — GOLD")
print("=" * 55)

# Inner join: teniamo solo ZIP code presenti in TUTTI e tre i dataset
# Se un ZIP manca in uno dei tre → viene scartato
economic_profile = (
    census_population
    .join(census_income, "zip_code", "inner")   # aggiunge median_income
    .join(zillow_rent, "zip_code", "inner")      # aggiunge market_rent
)

print(f"  ZIP con dati completi (NY totale): {economic_profile.count():,}")

🏆 STEP 3 — GOLD
  ZIP con dati completi (NY totale): 296


In [15]:
# ============================================================
# 5.3 GOLD — RENTAL STRESS INDEX
#
# Calcoliamo il "rent burden" per ogni ZIP code:
#   rent_burden_pct = (affitto_mensile * 12 / reddito_annuo) * 100
#
# Questo è lo standard HUD (Dept. of Housing and Urban Development):
#   < 30%  → Affordable   (spende meno del 30% del reddito in affitto)
#   30-50% → Stressed     (cost-burdened)
#   >= 50% → Severely Stressed (severely cost-burdened)
#
# È la metrica principale del progetto — identifica le "Stressed Areas"
# ============================================================

gold_market_analysis = (
    economic_profile
    .withColumn(
        "rent_burden_pct",
        when(
            col("median_income") > 0,
            spark_round(
                # market_rent è mensile → x12 per annualizzarlo
                # dividiamo per reddito annuo e moltiplichiamo x100 per %
                (col("market_rent") * 12 / col("median_income")) * 100, 2
            )
        ).otherwise(None)  # NULL se reddito = 0 (evita divisione per zero)
    )
    .withColumn(
        # Assegna categoria di stress in base alla soglia HUD
        "stress_category",
        when(col("rent_burden_pct") >= 50, "🔴 Severely Stressed")
        .when(col("rent_burden_pct") >= 30, "🟡 Stressed")
        .when(col("rent_burden_pct").isNotNull(), "🟢 Affordable")
        .otherwise("⚪ No Data")  # ZIP senza dati sufficienti
    )
)

print("📊 Distribuzione Rental Stress:")
gold_market_analysis.groupBy("stress_category") \
    .count() \
    .orderBy("stress_category") \
    .show()

📊 Distribuzione Rental Stress:
+--------------------+-----+
|     stress_category|count|
+--------------------+-----+
|🔴 Severely Stressed|   44|
|         🟡 Stressed|  142|
|       🟢 Affordable|  110|
+--------------------+-----+



In [17]:
# ============================================================
# 5.4 GOLD — AIRBNB BOROUGH SUMMARY
#
# Aggreghiamo i listing Airbnb per borough per capire:
#   - Quanti listing ci sono (listing concentration)
#   - Quanto sono occupati in media (occupancy pressure)
#   - Quanti sono "Entire home" (appartamenti sottratti al mercato)
#   - Quanti host hanno più listing (proxy host commerciali)
#
# Non abbiamo avg_price_per_night (NULL nel dataset NY)
# quindi lavoriamo solo con occupancy e concentrazione.
# ============================================================

airbnb_borough_summary = (
    airbnb_listings_enriched
    # Teniamo solo listing con dati di occupancy validi
    .filter(col("final_occupancy_rate").isNotNull())
    .groupBy("neighbourhood_group_cleansed")
    .agg(
        # Numero totale listing per borough
        count("listing_id").alias("num_listings"),

        # Occupancy media in percentuale (0-100%)
        spark_round(avg("final_occupancy_rate") * 100, 1).alias("avg_occupancy_pct"),

        # Media dei listing per host
        # Un valore alto indica presenza di host commerciali (multi-property)
        spark_round(avg("calculated_host_listings_count"), 1).alias("avg_host_listings"),

        # Numero di listing "Entire home/apt"
        # Questi sono appartamenti interi sottratti al mercato residenziale
        spark_sum(
            when(col("room_type") == "Entire home/apt", 1).otherwise(0)
        ).alias("entire_home_count")
    )
    .withColumn(
        # % di listing che sono appartamenti interi
        # Alto = più pressione sul mercato residenziale
        "entire_home_pct",
        spark_round(col("entire_home_count") / col("num_listings") * 100, 1)
    )
    .orderBy(col("num_listings").desc())
)

print("🏘️  Airbnb per Borough:")
airbnb_borough_summary.show(truncate=False)

🏘️  Airbnb per Borough:
+----------------------------+------------+-----------------+-----------------+-----------------+---------------+
|neighbourhood_group_cleansed|num_listings|avg_occupancy_pct|avg_host_listings|entire_home_count|entire_home_pct|
+----------------------------+------------+-----------------+-----------------+-----------------+---------------+
|Manhattan                   |15127       |58.4             |114.2            |9703             |64.1           |
|Brooklyn                    |12066       |62.6             |28.5             |5988             |49.6           |
|Queens                      |4592        |57.8             |42.3             |1651             |36.0           |
|Bronx                       |910         |50.7             |2.8              |345              |37.9           |
|Staten Island               |301         |40.3             |2.5              |142              |47.2           |
+----------------------------+------------+-----------------+---

In [18]:
# ============================================================
# 5.5 GOLD — AIRBNB PRESSURE INDEX
#
# Combina listing concentration e occupancy rate per calcolare
# un indice di pressione speculativa per borough.
#
# Formula:
#   pressure_score = (num_listings / max_listings) * avg_occupancy_pct
#
# Interpretazione:
#   - num_listings / max_listings → concentrazione normalizzata (0-1)
#   - * avg_occupancy_pct         → pesa per quanto sono effettivamente usati
#
# Borough con tanti listing molto occupati = alta pressione sul mercato
# ============================================================

# Calcoliamo il massimo numero di listing tra tutti i borough
max_listings = airbnb_borough_summary \
    .agg(spark_max("num_listings")) \
    .collect()[0][0]

airbnb_pressure = (
    airbnb_borough_summary
    .withColumn(
        "pressure_score",
        spark_round(
            (col("num_listings") / max_listings) * col("avg_occupancy_pct"), 2
        )
    )
    .orderBy(col("pressure_score").desc())
)

print("📊 Airbnb Pressure Index per Borough:")
airbnb_pressure.show(truncate=False)

📊 Airbnb Pressure Index per Borough:
+----------------------------+------------+-----------------+-----------------+-----------------+---------------+--------------+
|neighbourhood_group_cleansed|num_listings|avg_occupancy_pct|avg_host_listings|entire_home_count|entire_home_pct|pressure_score|
+----------------------------+------------+-----------------+-----------------+-----------------+---------------+--------------+
|Manhattan                   |15127       |58.4             |114.2            |9703             |64.1           |58.4          |
|Brooklyn                    |12066       |62.6             |28.5             |5988             |49.6           |49.93         |
|Queens                      |4592        |57.8             |42.3             |1651             |36.0           |17.55         |
|Bronx                       |910         |50.7             |2.8              |345              |37.9           |3.05          |
|Staten Island               |301         |40.3             

In [19]:
# ============================================================
# 5.6 GOLD — SALVATAGGIO SU MINIO
#
# Salviamo le tre tabelle Gold in formato Parquet su MinIO.
# coalesce(1) forza Spark a scrivere un singolo file Parquet
# invece di tanti file partizionati → più facile da leggere
# con pandas nella dashboard Streamlit.
# ============================================================
print("\n💾 Saving Gold layer to MinIO...")

# Stress economico per ZIP code → usato per la heatmap della dashboard
gold_market_analysis.write \
    .mode("overwrite") \
    .parquet(GOLD_PATH + "market_rental_stress/")
print("✅ market_rental_stress saved")

# Concentrazione Airbnb per borough → grafico a barre dashboard
airbnb_borough_summary.write \
    .mode("overwrite") \
    .parquet(GOLD_PATH + "airbnb_borough_summary/")
print("✅ airbnb_borough_summary saved")

# Pressure index → ranking borough per stress Airbnb
airbnb_pressure.write \
    .mode("overwrite") \
    .parquet(GOLD_PATH + "airbnb_pressure/")
print("✅ airbnb_pressure saved")

print("\n🎉 Gold layer saved to MinIO!\n")


💾 Saving Gold layer to MinIO...
✅ market_rental_stress saved
✅ airbnb_borough_summary saved
✅ airbnb_pressure saved

🎉 Gold layer saved to MinIO!



In [16]:
# ============================================================
# 6. OUTPUT FINALE — RISULTATI
#
# Stampiamo un riepilogo dei risultati principali della pipeline.
# Questi sono i numeri che presenteremo nella dashboard e
# nella relazione dell'esame.
# ============================================================
print("=" * 55)
print("📊 RISULTATI FINALI")
print("=" * 55)

# Distribuzione ZIP code per categoria di stress
total_zip  = gold_market_analysis.count()
affordable = gold_market_analysis.filter(col("rent_burden_pct") < 30).count()
stressed   = gold_market_analysis.filter(
    (col("rent_burden_pct") >= 30) & (col("rent_burden_pct") < 50)).count()
severe     = gold_market_analysis.filter(col("rent_burden_pct") >= 50).count()

print(f"\n📍 ZIP code NYC analizzati: {total_zip}")
print(f"   🟢 Affordable  (<30%): {affordable:,}")
print(f"   🟡 Stressed  (30-50%): {stressed:,}")
print(f"   🔴 Severely    (≥50%): {severe:,}")

# Top 10 ZIP code più stressati — le "Stressed Areas" del progetto
print("\n🔝 TOP 10 ZIP CODE PER RENTAL STRESS:")
gold_market_analysis \
    .select(
        "zip_code",
        "total_population",
        "median_income",
        "market_rent",
        "rent_burden_pct",
        "stress_category"
    ) \
    .orderBy(col("rent_burden_pct").desc()) \
    .show(10, truncate=False)

# Top 5 ZIP più accessibili — per bilanciare l'analisi
print("\n🟢 TOP 5 ZIP PIÙ ACCESSIBILI:")
gold_market_analysis \
    .select("zip_code", "median_income", "market_rent", "rent_burden_pct") \
    .filter(col("rent_burden_pct").isNotNull()) \
    .orderBy(col("rent_burden_pct").asc()) \
    .show(5, truncate=False)

# Pressione Airbnb per borough
print("\n🏘️  AIRBNB PRESSURE INDEX PER BOROUGH:")
airbnb_pressure.show(truncate=False)

# Dettaglio per tipo di stanza
print("\n🏘️  AIRBNB: DETTAGLIO PER ROOM TYPE:")
airbnb_listings_enriched \
    .filter(col("final_occupancy_rate").isNotNull()) \
    .groupBy("room_type") \
    .agg(
        count("id").alias("num_listings"),
        spark_round(avg("final_occupancy_rate") * 100, 1).alias("avg_occupancy_pct")
    ) \
    .orderBy(col("num_listings").desc()) \
    .show(truncate=False)

📊 RISULTATI FINALI

📍 ZIP code NYC analizzati: 353
   🟢 Affordable  (<30%): 144
   🟡 Stressed  (30-50%): 157
   🔴 Severely    (≥50%): 52

🔝 TOP 10 ZIP CODE PER RENTAL STRESS:
+--------+----------------+-------------+-----------+---------------+--------------------+
|zip_code|total_population|median_income|market_rent|rent_burden_pct|stress_category     |
+--------+----------------+-------------+-----------+---------------+--------------------+
|11976   |2952            |180250       |107500.0   |715.67         |🔴 Severely Stressed|
|11959   |734             |88542        |45000.0    |609.88         |🔴 Severely Stressed|
|11932   |725             |173672       |75875.0    |524.26         |🔴 Severely Stressed|
|11930   |842             |123257       |47073.89   |458.3          |🔴 Severely Stressed|
|11978   |4705            |125179       |47266.668  |453.11         |🔴 Severely Stressed|
|11937   |21882           |129883       |42870.51   |396.08         |🔴 Severely Stressed|
|11963   |83

In [21]:
from pyspark.sql.functions import explode, col, avg, count, round as spark_round, when, sum as spark_sum
from sedona.spark import SedonaContext
from sedona.sql.st_constructors import ST_Point
from sedona.sql.st_predicates import ST_Within

# Registra Sedona sulla SparkSession esistente
sedona = SedonaContext.create(spark)
print("✅ Sedona ready")

✅ Sedona ready


In [18]:
import requests, os

GEOJSON_PATH = "/tmp/nyc_zip.geojson"

if not os.path.exists(GEOJSON_PATH):
    print("⬇️  Downloading NYC ZIP GeoJSON...")
    url = "https://data.cityofnewyork.us/resource/pri4-ifjk.geojson?$limit=5000"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(GEOJSON_PATH, "wb") as f:
        f.write(r.content)
    print(f"✅ GeoJSON scaricato ({os.path.getsize(GEOJSON_PATH)//1024} KB)")
else:
    print("⏭️  GeoJSON già presente")

⬇️  Downloading NYC ZIP GeoJSON...
✅ GeoJSON scaricato (3076 KB)


In [22]:
# Legge il GeoJSON dei confini ZIP di NYC come Spark DataFrame
gdf_zip_raw = spark.read \
    .format("geojson") \
    .option("multiLine", "true") \
    .load(GEOJSON_PATH)

features = gdf_zip_raw.select(explode(col("features")).alias("feature"))
features.printSchema()

root
 |-- feature: struct (nullable = true)
 |    |-- geometry: geometry (nullable = true)
 |    |-- properties: struct (nullable = true)
 |    |    |-- label: string (nullable = true)
 |    |    |-- modzcta: string (nullable = true)
 |    |    |-- pop_est: string (nullable = true)
 |    |    |-- zcta: string (nullable = true)
 |    |-- type: string (nullable = true)



In [23]:
from pyspark.sql.functions import explode, col
# Estrae zip_code e geometria dalla struct
gdf_zip = features.select(
    col("feature.properties.modzcta").alias("zip_code"),  # ZIP code
    col("feature.geometry").alias("geometry")              # poligono del ZIP
).filter(col("zip_code").isNotNull())

print(f"  ZIP poligoni caricati: {gdf_zip.count():,}")
gdf_zip.show(5)

  ZIP poligoni caricati: 178
+--------+--------------------+
|zip_code|            geometry|
+--------+--------------------+
|   10001|MULTIPOLYGON (((-...|
|   10002|MULTIPOLYGON (((-...|
|   10003|MULTIPOLYGON (((-...|
|   10026|MULTIPOLYGON (((-...|
|   10004|MULTIPOLYGON (((-...|
+--------+--------------------+
only showing top 5 rows



In [24]:
from sedona.sql.st_predicates import ST_Within
from sedona.sql.st_constructors import ST_Point

# Crea colonna geometria punto da lat/long di ogni listing Airbnb
airbnb_geo = (
    airbnb_listings_enriched
    .filter(col("latitude").isNotNull())
    .filter(col("longitude").isNotNull())
    .withColumn(
        # ST_Point(longitude, latitude) — attenzione all'ordine!
        "geometry",
        ST_Point(col("longitude"), col("latitude"))
    )
)

print(f"  Listing con coordinate: {airbnb_geo.count():,}")

# Spatial join: assegna ogni punto Airbnb al suo ZIP code
# ST_Within(punto, poligono) = True se il punto è dentro il poligono
airbnb_with_zip = (
    airbnb_geo.alias("a")
    .join(
        gdf_zip.alias("z"),
        ST_Within(col("a.geometry"), col("z.geometry")),
        "left"
    )
    .select(
        col("a.id"),
        col("a.neighbourhood_group_cleansed"),
        col("a.neighbourhood_cleansed"),
        col("a.latitude"),
        col("a.longitude"),
        col("a.final_occupancy_rate"),
        col("a.room_type"),
        col("a.calculated_host_listings_count"),
        col("z.zip_code")
    )
)

matched = airbnb_with_zip.filter(col("zip_code").isNotNull()).count()
total = airbnb_with_zip.count()
print(f"  Listing totali:          {total:,}")
print(f"  Listing assegnati a ZIP: {matched:,}")
airbnb_with_zip.show(5)

  Listing con coordinate: 36,445
  Listing totali:          36,445
  Listing assegnati a ZIP: 36,440
+----+----------------------------+----------------------+--------+---------+--------------------+---------------+------------------------------+--------+
|  id|neighbourhood_group_cleansed|neighbourhood_cleansed|latitude|longitude|final_occupancy_rate|      room_type|calculated_host_listings_count|zip_code|
+----+----------------------------+----------------------+--------+---------+--------------------+---------------+------------------------------+--------+
|2595|                   Manhattan|               Midtown|40.75356|-73.98559|                NULL|Entire home/apt|                             3|   10018|
|6848|                    Brooklyn|          Williamsburg|40.70935|-73.95342|              0.6575|Entire home/apt|                             1|   11211|
|6872|                   Manhattan|           East Harlem|40.80107|-73.94255|              0.1644|   Private room|          

In [25]:
# ============================================================
# SPATIAL ANALYSIS — JOIN CON STRESS INDEX
#
# Uniamo i listing Airbnb (con ZIP code dal spatial join)
# con lo stress index calcolato nel Gold layer.
# Aggreghiamo per ZIP code per ottenere la tabella finale
# che alimenterà la dashboard.
# ============================================================

# Join tra listing Airbnb e stress index per ZIP
complete = (
    airbnb_with_zip
    .join(
        gold_market_analysis.select(
            "zip_code", "median_income", "market_rent",
            "total_population", "rent_burden_pct", "stress_category"
        ),
        "zip_code", "left"
    )
)

# Aggregazione per ZIP code
zip_airbnb_stress = (
    complete
    .groupBy(
        "zip_code",
        "neighbourhood_group_cleansed",   # borough
        "rent_burden_pct",
        "stress_category",
        "median_income",
        "market_rent",
        "total_population"
    )
    .agg(
        # Numero listing Airbnb nel ZIP
        count("id").alias("num_airbnb_listings"),
        # Occupancy media nel ZIP (solo listing con dati validi)
        spark_round(avg("final_occupancy_rate") * 100, 1).alias("avg_occupancy_pct"),
        # Media listing per host nel ZIP (proxy commercializzazione)
        spark_round(avg("calculated_host_listings_count"), 1).alias("avg_host_listings"),
        # % listing "Entire home" nel ZIP
        spark_round(
            spark_sum(when(col("room_type") == "Entire home/apt", 1).otherwise(0)) /
            count("id") * 100, 1
        ).alias("entire_home_pct")
    )
    .orderBy(col("rent_burden_pct").desc())
)

print(f"  ZIP analizzati: {zip_airbnb_stress.count():,}")
zip_airbnb_stress.show(10, truncate=False)

  ZIP analizzati: 183
+--------+----------------------------+---------------+--------------------+-------------+-----------+----------------+-------------------+-----------------+-----------------+---------------+
|zip_code|neighbourhood_group_cleansed|rent_burden_pct|stress_category     |median_income|market_rent|total_population|num_airbnb_listings|avg_occupancy_pct|avg_host_listings|entire_home_pct|
+--------+----------------------------+---------------+--------------------+-------------+-----------+----------------+-------------------+-----------------+-----------------+---------------+
|10454   |Bronx                       |147.17         |🔴 Severely Stressed|24086        |2954.0105  |39570           |56                 |40.4             |1.8              |35.7           |
|10002   |Manhattan                   |107.77         |🔴 Severely Stressed|48386        |4345.4795  |76873           |810                |41.0             |45.7             |62.0           |
|10456   |Bronx     

In [26]:
# ============================================================
# SALVATAGGIO SPATIAL ANALYSIS SU MINIO
# ============================================================
print("💾 Saving spatial analysis to MinIO...")

# Tabella principale per la dashboard — una riga per ZIP code
# contiene sia i dati Airbnb che lo stress index economico
zip_airbnb_stress.write \
    .mode("overwrite") \
    .parquet(GOLD_PATH + "zip_airbnb_stress_summary/")
print("✅ zip_airbnb_stress_summary saved")

# Salva anche i listing completi con ZIP code assegnato
# utile per la heatmap puntuale della dashboard
airbnb_with_zip.write \
    .mode("overwrite") \
    .parquet(GOLD_PATH + "airbnb_listings_with_zip/")
print("✅ airbnb_listings_with_zip saved")

print("\n🎉 Spatial analysis saved to MinIO!")

💾 Saving spatial analysis to MinIO...
✅ zip_airbnb_stress_summary saved
✅ airbnb_listings_with_zip saved

🎉 Spatial analysis saved to MinIO!
